# Deepfake Detection — Pipeline nhóm (dataset `project_data`)

**Luồng xử lý:** ảnh → resize 256×256 (xám) → **ảnh dư PCA** → đặc trưng **FFT + LBP + Noise** → ghép thành vector **187 chiều** → chuẩn hóa → chia train/val/test → **SVM / RF / GBM**.

| Người | Nhiệm vụ | Cell |
|---|---|---|
| **A — Data** | Lấy `project_data`, giải nén, resize/normalize, chia 80/10/10 | Setup + Data |
| **B — Features** | FFT + LBP + Noise → ghép feature vector + định nghĩa nhóm đặc trưng cho ablation | `[B] ...` |
| **C — Model** | Train SVM/RF/GBM, đánh giá AUC/F1/Confusion, ablation, vẽ biểu đồ | `[C] ...` |

> Chạy tuần tự từ trên xuống. Phần nặng nhất là vòng trích đặc trưng (PCA theo từng ảnh). Sau khi người B chạy xong sẽ có `train.pkl / val.pkl / test.pkl / scaler.pkl` để bàn giao cho người C.


In [ ]:
# [A] Cài thư viện cần thiết
!pip install scikit-image tqdm -q


In [ ]:
# ============================================================
# [NGƯỜI A] LẤY DỮ LIỆU project_data (40.000 ảnh) & GIẢI NÉN
#   - Ưu tiên đọc từ Google Drive đã mount; nếu không thấy thì tải bằng gdown.
# ============================================================
import os, zipfile

DATA_DIR = '/content/project_data'
ZIP_OUT  = '/content/project_data.zip'
FILE_ID  = '1hKCgB5upHkUhb-baWL_2zN74JQoYC7A4'   # project_data.zip (bản sao trên Drive)

if not os.path.isdir(DATA_DIR):
    zip_path = None

    # 1) Thử lấy từ Google Drive đã mount
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        candidates = [
            '/content/drive/MyDrive/Deepfake_project/project_data.zip',
            '/content/drive/Shareddrives/Deepfake_project/project_data.zip',
        ]
        zip_path = next((p for p in candidates if os.path.exists(p)), None)
    except Exception as e:
        print('[INFO] Bỏ qua mount Drive:', e)

    # 2) Không thấy trong Drive -> tải trực tiếp bằng gdown
    if zip_path is None:
        import gdown
        print('[INFO] Không thấy ZIP trong Drive -> tải bằng gdown...')
        gdown.download(f'https://drive.google.com/uc?id={FILE_ID}', ZIP_OUT, quiet=False)
        zip_path = ZIP_OUT

    print('[INFO] Đang giải nén:', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
else:
    print('[OK] Đã có sẵn', DATA_DIR)

num_real = len(os.listdir(os.path.join(DATA_DIR, 'real')))
num_fake = len(os.listdir(os.path.join(DATA_DIR, 'fake')))
print(f'[THÀNH CÔNG] Real: {num_real} ảnh | Fake: {num_fake} ảnh')


In [ ]:
# ========================================================
# QUY TRÌNH TIỀN XỬ LÝ: ĐỌC ẢNH, RESIZE VÀ TRÍCH XUấT ẢNH DƯ PCA
# ========================================================

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# --- ĐỊNH NGHĨA HÀM TRÍCH XUấT ẢNH DƯ PCA CHUẨN BASELINE ---
def get_pca_residual_image(img_gray, num_components_to_remove=32):
    # Đảm bảo ảnh ở dạng float64 phục vụ toán học ma trận
    img_data = img_gray.astype(np.float64)

    # Khởi tạo thuật toán PCA
    pca = PCA()
    pca.fit(img_data)

    # Biến đổi ảnh sang không gian các thành phần chính
    img_transformed = pca.transform(img_data)

    # Tạo bản sao và loại bỏ Top N thành phần tương quan cao (High-correlation signals)
    img_transformed_residual = img_transformed.copy()
    img_transformed_residual[:, :num_components_to_remove] = 0

    # Nghịch đảo ma trận để tái cấu trúc lại thành ảnh dư (Residual Image)
    residual_data = pca.inverse_transform(img_transformed_residual)

    # Chuẩn hóa giá trị pixel về đoạn [0, 255] định dạng uint8
    residual_img_8u = cv2.convertScaleAbs(residual_data)

    return residual_img_8u


# --- TIỀN XỬ LÝ VÀ CHẠY THỬ NGHIỆM TRÊN ẢNH MẪU ---
print("--- Đang tiến hành tiền xử lý và trích xuất ảnh dư PCA ---")

# Lấy danh sách file ảnh thật và giả
real_files = [f for f in os.listdir('/content/project_data/real') if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
fake_files = [f for f in os.listdir('/content/project_data/fake') if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

if len(real_files) > 0 and len(fake_files) > 0:
    real_sample_path = os.path.join('/content/project_data/real', real_files[0])
    fake_sample_path = os.path.join('/content/project_data/fake', fake_files[0])

    # 1. Đọc ảnh xám (Grayscale)
    img_real_raw = cv2.imread(real_sample_path, cv2.IMREAD_GRAYSCALE)
    img_fake_raw = cv2.imread(fake_sample_path, cv2.IMREAD_GRAYSCALE)

    # 2. Thay đổi kích thước về độ phân giải chuẩn 256x256 để tối ưu tài nguyên
    img_real_resized = cv2.resize(img_real_raw, (256, 256))
    img_fake_resized = cv2.resize(img_fake_raw, (256, 256))

    # 3. Trích xuất ảnh dư bằng cách loại bỏ N=32 thành phần chính (Theo Mục 6 trong Paper)
    N_components = 32
    residual_real = get_pca_residual_image(img_real_resized, num_components_to_remove=N_components)
    residual_fake = get_pca_residual_image(img_fake_resized, num_components_to_remove=N_components)

    # --- TRỰC QUAN HÓA KẾT QUẢ ĐẦU RA ---
    plt.figure(figsize=(14, 8))

    # Ảnh thật gốc (đã resize)
    plt.subplot(2, 2, 1)
    plt.imshow(img_real_resized, cmap='gray')
    plt.title("1. Ảnh THẬT nguyên bản (256x256 Grayscale)")
    plt.axis('off')

    # Ảnh dư của ảnh thật (Tín hiệu tương quan thấp)
    plt.subplot(2, 2, 2)
    plt.imshow(residual_real, cmap='gray')
    plt.title(f"2. Ảnh dư Real (Đã loại bỏ Top {N_components} PCA)")
    plt.axis('off')

    # Ảnh giả gốc (đã resize)
    plt.subplot(2, 2, 3)
    plt.imshow(img_fake_resized, cmap='gray')
    plt.title("3. Ảnh GIẢ nguyên bản (256x256 Grayscale)")
    plt.axis('off')

    # Ảnh dư của ảnh giả (Tín hiệu tương quan thấp)
    plt.subplot(2, 2, 4)
    plt.imshow(residual_fake, cmap='gray')
    plt.title(f"4. Ảnh dư Fake (Đã loại bỏ Top {N_components} PCA)")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"\n[THÀNH CÔNG] Tiền xử lý hoàn tất!")
    print(f" -> Kích thước ma trận ảnh dư Real: {residual_real.shape}")
    print(f" -> Kích thước ma trận ảnh dư Fake: {residual_fake.shape}")
else:
    print("[LỖI] Không tìm thấy ảnh trong thư mục /content/project_data/. Hãy chắc chắn bạn đã giải nén thành công dữ liệu ở Ô số 1.")

In [ ]:
# ============================================================
# [NGƯỜI B] ĐẶC TRƯNG: FFT + LBP + NOISE  ->  GHÉP FEATURE VECTOR
#   Trích trên "ảnh dư PCA" (residual) để làm nổi dấu vết của ảnh giả.
# ============================================================
import os
import numpy as np
import cv2
from skimage.feature import local_binary_pattern
from tqdm import tqdm

# --- Cấu hình ---
N_components = 32                       # số thành phần PCA loại bỏ (lấy ảnh dư)
LBP_P, LBP_R = 8, 1                     # tham số LBP (8 điểm, bán kính 1)
FFT_BINS    = 64
LBP_BINS    = LBP_P * (LBP_P - 1) + 3   # = 59 mẫu của method='nri_uniform'
NOISE_BINS  = 64
FEATURE_DIM = FFT_BINS + LBP_BINS + NOISE_BINS   # = 187

# --- 1) FFT: phân bố năng lượng phổ tần số (ảnh giả thường lệch ở tần số cao) ---
def extract_fft(gray):
    f   = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.log1p(np.abs(f))
    hist, _ = np.histogram(mag, bins=FFT_BINS, range=(mag.min(), mag.max()))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

# --- 2) LBP: kết cấu cục bộ.
#     SỬA: dùng 'nri_uniform' (59 mẫu) cho khớp LBP_BINS=59.
#     Bản cũ dùng 'uniform' (chỉ 10 mẫu) nên 49/59 bin luôn = 0 (đặc trưng chết). ---
def extract_lbp(gray):
    lbp = local_binary_pattern(gray, P=LBP_P, R=LBP_R, method='nri_uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=LBP_BINS, range=(0, LBP_BINS))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

# --- 3) Noise residual: thống kê nhiễu tần số cao (fingerprint của GAN) ---
def extract_noise(gray):
    noise = gray.astype(np.float32) - cv2.GaussianBlur(gray, (5, 5), 0).astype(np.float32)
    hist, _ = np.histogram(noise, bins=NOISE_BINS, range=(-50, 50))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

# --- Ghép 3 nhóm thành 1 vector: [ FFT | LBP | Noise ] ---
def extract_features(gray):
    return np.concatenate([extract_fft(gray), extract_lbp(gray), extract_noise(gray)])

# Vị trí (cột) từng nhóm trong feature vector -> dùng cho ABLATION của người C
feature_groups = {
    'fft':   (0,                        FFT_BINS),
    'lbp':   (FFT_BINS,                 FFT_BINS + LBP_BINS),
    'noise': (FFT_BINS + LBP_BINS,      FEATURE_DIM),
}
feature_names = ([f'fft_{i}'   for i in range(FFT_BINS)] +
                 [f'lbp_{i}'   for i in range(LBP_BINS)] +
                 [f'noise_{i}' for i in range(NOISE_BINS)])

# --- Quét toàn bộ dataset, trích đặc trưng trên ảnh dư PCA ---
DATA_DIR = '/content/project_data'
X_all, y_all, paths_all = [], [], []
for label_name, label_idx in [('real', 0), ('fake', 1)]:
    folder = os.path.join(DATA_DIR, label_name)
    files  = sorted([f for f in os.listdir(folder)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    print(f'[{label_name.upper()}] {len(files)} ảnh')
    for fname in tqdm(files, desc=label_name):
        img = cv2.imread(os.path.join(folder, fname), cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        img = cv2.resize(img, (256, 256))
        residual = get_pca_residual_image(img, num_components_to_remove=N_components)
        X_all.append(extract_features(residual))
        y_all.append(label_idx)
        paths_all.append(os.path.join(folder, fname))

X_all = np.asarray(X_all, dtype=np.float32)
y_all = np.asarray(y_all, dtype=np.int32)
assert X_all.shape[1] == FEATURE_DIM, f'Sai chiều đặc trưng: {X_all.shape[1]} != {FEATURE_DIM}'
print(f'\nTổng: {len(y_all)} ảnh | real={(y_all==0).sum()} fake={(y_all==1).sum()} | dim={X_all.shape[1]}')


In [ ]:
# ============================================================
# [NGƯỜI B] CHIA TRAIN/VAL/TEST (80/10/10) + CHUẨN HÓA + LƯU BÀN GIAO
# ============================================================
import os, pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

OUTPUT_DIR = '/content/features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Chia phân tầng theo nhãn (giữ tỉ lệ real/fake ở cả 3 tập)
idx = np.arange(len(y_all))
idx_train, idx_tmp = train_test_split(idx,      test_size=0.2, stratify=y_all,           random_state=42)
idx_val,   idx_test = train_test_split(idx_tmp, test_size=0.5, stratify=y_all[idx_tmp],  random_state=42)

# Chuẩn hóa: CHỈ fit trên train rồi transform val/test (tránh rò rỉ dữ liệu)
scaler = StandardScaler().fit(X_all[idx_train])
config = {'img_size': 256, 'pca_remove': N_components, 'feature_dim': FEATURE_DIM,
          'lbp_method': 'nri_uniform', 'split': '80/10/10'}

for name, ids in [('train', idx_train), ('val', idx_val), ('test', idx_test)]:
    Xs = scaler.transform(X_all[ids]).astype(np.float32)
    ys = y_all[ids]
    ps = [paths_all[i] for i in ids]
    with open(f'{OUTPUT_DIR}/{name}.pkl', 'wb') as f:
        pickle.dump({'X': Xs, 'y': ys, 'paths': ps,
                     'feature_names': feature_names,
                     'feature_groups': feature_groups,
                     'config': config}, f)
    print(f'  {name}: {len(ys)} ảnh | real={(ys==0).sum()} fake={(ys==1).sum()} | shape={Xs.shape} -> {name}.pkl')

with open(f'{OUTPUT_DIR}/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('\n✅ XONG phần đặc trưng (người B).')
print('   Bàn giao người C: train.pkl, val.pkl, test.pkl, scaler.pkl')
print('   feature_groups =', feature_groups, '-> người C dùng để ablation.')


In [ ]:
# [B -> C] (Tùy chọn) Lưu các file đặc trưng lên Drive để bàn giao
import shutil, os

DST = '/content/drive/MyDrive/Deepfake_project/features'
os.makedirs(DST, exist_ok=True)

for fname in ['train.pkl', 'val.pkl', 'test.pkl', 'scaler.pkl']:
    shutil.copy(f'/content/features/{fname}', f'{DST}/{fname}')
    print('Đã lưu:', fname)

print('Xong! File đặc trưng đã an toàn trên Drive.')


## [Người C] Huấn luyện & đánh giá mô hình

Nạp đặc trưng do người B bàn giao, huấn luyện **SVM / Random Forest / Gradient Boosting**, đánh giá bằng **Accuracy / F1 / AUC / Confusion matrix**, chạy **ablation** (đo đóng góp từng nhóm đặc trưng) và **vẽ biểu đồ** kết quả.


In [ ]:
# ============================================================
# [NGƯỜI C] NẠP ĐẶC TRƯNG ĐỂ HUẤN LUYỆN
# ============================================================
import pickle, numpy as np

# Chạy cùng phiên với người B -> đọc ở /content/features.
# Nếu mở phiên mới: đổi thành '/content/drive/MyDrive/Deepfake_project/features'
FEAT_DIR = '/content/features'

def load_split(name):
    with open(f'{FEAT_DIR}/{name}.pkl', 'rb') as f:
        return pickle.load(f)

train_d, val_d, test_d = load_split('train'), load_split('val'), load_split('test')
X_train, y_train = train_d['X'], train_d['y']
X_val,   y_val   = val_d['X'],   val_d['y']
X_test,  y_test  = test_d['X'],  test_d['y']
feature_groups   = train_d['feature_groups']   # {'fft':(0,64), 'lbp':(64,123), 'noise':(123,187)}

print('train:', X_train.shape, '| val:', X_val.shape, '| test:', X_test.shape)
print('feature_groups:', feature_groups)


In [ ]:
# ============================================================
# [NGƯỜI C] HUẤN LUYỆN & ĐÁNH GIÁ: SVM, RANDOM FOREST, GBM
#   Đặc trưng đã được StandardScaler ở bước B -> train trực tiếp.
# ============================================================
import numpy as np, pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# RBF-SVM chậm khi dữ liệu lớn -> lấy mẫu con để train SVM cho nhanh.
# Đặt SVM_MAX_TRAIN = None nếu muốn train SVM trên TOÀN BỘ tập train.
SVM_MAX_TRAIN = 8000

models = {
    'SVM (RBF)':         SVC(kernel='rbf', C=10, gamma='scale', cache_size=1000, random_state=42),
    'Random Forest':     RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

def get_score(model, X):
    # xác suất lớp 1 nếu có, ngược lại dùng decision_function (SVM) -> phục vụ AUC
    return model.predict_proba(X)[:, 1] if hasattr(model, 'predict_proba') else model.decision_function(X)

def evaluate(model, X, y):
    pred  = model.predict(X)
    score = get_score(model, X)
    return {'acc': accuracy_score(y, pred), 'f1': f1_score(y, pred),
            'auc': roc_auc_score(y, score), 'cm': confusion_matrix(y, pred)}

rows, trained, test_scores, test_cms = [], {}, {}, {}
for name, model in models.items():
    print(f'\n=== {name} ===')
    if name.startswith('SVM') and SVM_MAX_TRAIN and len(y_train) > SVM_MAX_TRAIN:
        rs = np.random.RandomState(42)
        sub = rs.choice(len(y_train), SVM_MAX_TRAIN, replace=False)
        model.fit(X_train[sub], y_train[sub])
        print(f'  (train SVM trên {SVM_MAX_TRAIN}/{len(y_train)} mẫu)')
    else:
        model.fit(X_train, y_train)

    trained[name] = model
    val_m, test_m = evaluate(model, X_val, y_val), evaluate(model, X_test, y_test)
    test_scores[name], test_cms[name] = get_score(model, X_test), test_m['cm']
    print(f'  VAL : acc={val_m["acc"]:.4f}  f1={val_m["f1"]:.4f}  auc={val_m["auc"]:.4f}')
    print(f'  TEST: acc={test_m["acc"]:.4f}  f1={test_m["f1"]:.4f}  auc={test_m["auc"]:.4f}')
    print(f'  Confusion (test) [ [TN FP] [FN TP] ]:\n{test_m["cm"]}')
    rows.append({'model': name,
                 'val_acc': val_m['acc'], 'val_f1': val_m['f1'], 'val_auc': val_m['auc'],
                 'test_acc': test_m['acc'], 'test_f1': test_m['f1'], 'test_auc': test_m['auc']})

results = pd.DataFrame(rows).set_index('model').round(4)
print('\n===== BẢNG SO SÁNH MÔ HÌNH =====')
print(results)


In [ ]:
# ============================================================
# [NGƯỜI C] ABLATION STUDY: đo đóng góp của TỪNG nhóm đặc trưng
#   Dùng feature_groups (người B) để bỏ / chỉ giữ từng nhóm rồi train lại.
#   Dùng Random Forest cho nhanh và ổn định.
# ============================================================
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def cols(*group_names):
    idx = []
    for g in group_names:
        s, e = feature_groups[g]
        idx.extend(range(s, e))
    return np.array(idx)

variants = {
    'full':       cols('fft', 'lbp', 'noise'),
    'no_fft':     cols('lbp', 'noise'),
    'no_lbp':     cols('fft', 'noise'),
    'no_noise':   cols('fft', 'lbp'),
    'only_fft':   cols('fft'),
    'only_lbp':   cols('lbp'),
    'only_noise': cols('noise'),
}

rows = []
for vname, c in variants.items():
    clf = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
    clf.fit(X_train[:, c], y_train)
    proba = clf.predict_proba(X_test[:, c])[:, 1]
    pred  = clf.predict(X_test[:, c])
    rows.append({'variant': vname, 'n_features': len(c),
                 'test_acc': accuracy_score(y_test, pred),
                 'test_f1':  f1_score(y_test, pred),
                 'test_auc': roc_auc_score(y_test, proba)})

abl = pd.DataFrame(rows).set_index('variant').round(4)
abl['delta_auc_vs_full'] = (abl['test_auc'] - abl.loc['full', 'test_auc']).round(4)
print('===== BẢNG ABLATION (RandomForest, đánh giá trên TEST) =====')
print(abl)
print('\nGhi chú: delta_auc_vs_full < 0 khi BỎ một nhóm => nhóm đó đóng góp tích cực.')


In [ ]:
# ============================================================
# [NGƯỜI C] VẼ BIỂU ĐỒ: ROC, Confusion matrix, Ablation
# ============================================================
import numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, ConfusionMatrixDisplay

# 1) ROC các mô hình trên test
plt.figure(figsize=(6, 5))
for name in trained:
    fpr, tpr, _ = roc_curve(y_test, test_scores[name])
    plt.plot(fpr, tpr, label=f'{name} (AUC={results.loc[name, "test_auc"]:.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC trên tập TEST'); plt.legend(); plt.tight_layout(); plt.show()

# 2) Confusion matrix từng mô hình
fig, axes = plt.subplots(1, len(trained), figsize=(5 * len(trained), 4))
for ax, name in zip(np.atleast_1d(axes), trained):
    ConfusionMatrixDisplay(test_cms[name], display_labels=['real', 'fake']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout(); plt.show()

# 3) Ablation: AUC theo từng biến thể đặc trưng
plt.figure(figsize=(8, 4))
order = abl.sort_values('test_auc')
plt.barh(order.index, order['test_auc'])
plt.xlabel('AUC trên TEST'); plt.title('Ablation: đóng góp của các nhóm đặc trưng')
plt.xlim(min(0.5, order['test_auc'].min() - 0.02), 1.0)
plt.tight_layout(); plt.show()
